# Should you cheat? — Push-T without diffusion

*PyData Boston × Cursor hackathon, Moderna HQ, 2026-05-13.*

Push-T was meant to be solved with **image-only diffusion policies** (Chi et al., 2024). But diffusion policies don't fit on a single laptop. So this notebook cheats: instead of learning the T's pose from pixels with a giant model, we **feature-engineer** it — fitting the canonical T's `(x, y, θ, scale)` directly to the green channel using Powell optimization on a coverage loss. Perception becomes a 4-parameter fit, and the rest of Push-T becomes tractable.


In [ ]:
import marimo as mo

# pusht — Diffusion Policy benchmark

2D push manipulation task from the [Diffusion Policy paper](https://arxiv.org/abs/2303.04137)
(Chi et al., 2024). The "blue ball" agent pushes a teal T-shape until it
overlaps the gray goal T.

- **206 episodes**, **25,650 frames** total, 10 FPS, 96×96 RGB
- 2D state (pusher x/y), 2D action (target x/y), scalar reward
- T-shape pose is **not** in the parquet — it's only in the image


In [ ]:
from huggingface_hub import hf_hub_download
import av
from PIL import Image

In [ ]:
def sample_frames(mp4_path, n=6):
        """Decode `n` frames evenly spaced through the MP4."""
        container = av.open(mp4_path)
        stream = container.streams.video[0]
        total = stream.frames
        targets = {int(total * i / n) for i in range(n)}
        out = []
        for i, frame in enumerate(container.decode(stream)):
            if i in targets:
                out.append((i, frame.to_image()))
                if len(out) >= n:
                    break
        container.close()
        fps = float(stream.average_rate) if stream.average_rate else 0.0
        return out, (stream.width, stream.height, fps, total)
def show_frames(frames, info, label):
        """Render frames as a horizontal strip of captioned images."""
        w, h, fps, total = info
        header = mo.md(f"**{label}** — {w}x{h}, {total} frames total")
        thumbs = [
            mo.vstack([mo.image(img, width=180), mo.md(f"frame {idx}")])
            for idx, img in frames
        ]
        return mo.vstack([header, mo.hstack(thumbs, wrap=True, gap=1)])

## Sample frames from the dataset


In [ ]:
pusht_mp4 = hf_hub_download(
        repo_id="lerobot/pusht",
        filename="videos/observation.image/chunk-000/file-000.mp4",
        repo_type="dataset",
    )
pusht_frames, pusht_info = sample_frames(pusht_mp4, n=6)
show_frames(pusht_frames, pusht_info, "lerobot/pusht")

## Full video + scrubbable still


In [ ]:
pusht_video_url = "https://huggingface.co/datasets/lerobot/pusht/resolve/main/videos/observation.image/chunk-000/file-000.mp4"
mo.video(src=pusht_video_url, controls=True, muted=True, width=480)

In [ ]:
import functools
def decode_frame(path, frame_idx):
        """Decode a single frame from MP4 by index. Cached so the slider feels fast."""
        container = av.open(path)
        stream = container.streams.video[0]
        img = None
        for i, frame in enumerate(container.decode(stream)):
            if i == frame_idx:
                img = frame.to_image()
                break
        container.close()
        return img
pusht_total_frames = 25650
pusht_frame_slider = mo.ui.slider(
        0, pusht_total_frames - 1, value=0, step=1,
        label="frame index", show_value=True, full_width=True,
    )
pusht_frame_slider

In [ ]:
pusht_current_frame = decode_frame(pusht_mp4, int(pusht_frame_slider.value))
mo.vstack([
        mo.md(f"### Frame {int(pusht_frame_slider.value)} / {pusht_total_frames - 1}"),
        mo.hstack([
            mo.vstack([
                mo.md("**Live video player**"),
                mo.video(src=pusht_video_url, controls=True, muted=True, width=360),
            ]),
            mo.vstack([
                mo.md("**Decoded still (Python)**"),
                mo.image(pusht_current_frame, width=360),
            ]),
        ], gap=2, justify="start"),
    ])

In [ ]:
import numpy as np
from scipy.ndimage import rotate as nd_rotate, binary_dilation, label, zoom as nd_zoom
from scipy.optimize import minimize
import pyarrow.parquet as pq
def make_canonical_t(cw=24, ch=6, sw=5, sh=14):
        H, W = ch + sh, cw
        t = np.zeros((H, W), dtype=bool)
        cx = W // 2
        t[0:ch, :] = True
        t[ch:ch+sh, cx - sw//2 : cx - sw//2 + sw] = True
        return t
CANONICAL_T = make_canonical_t()
def green_mask(frame):
        arr = np.array(frame)
        r, g, b = arr[..., 0].astype(int), arr[..., 1].astype(int), arr[..., 2].astype(int)
        return (g - r > 40) & (g - b > 40)
def render_t(canvas_shape, cx, cy, theta_deg, scale):
        """Render canonical T at (cx, cy, theta, scale)."""
        t = CANONICAL_T.astype(float)
        if scale != 1.0 and scale > 0:
            t = nd_zoom(t, scale, order=0)
        t_rot = nd_rotate(t, theta_deg, order=0, reshape=True) > 0.5
        H, W = canvas_shape
        th, tw = t_rot.shape
        x0 = int(round(cx - tw / 2))
        y0 = int(round(cy - th / 2))
        out = np.zeros((H, W), dtype=bool)
        sx0 = max(0, -x0); sy0 = max(0, -y0)
        dx0 = max(0, x0); dy0 = max(0, y0)
        sx1 = min(tw, W - x0); sy1 = min(th, H - y0)
        if sx1 > sx0 and sy1 > sy0:
            out[dy0:dy0+sy1-sy0, dx0:dx0+sx1-sx0] = t_rot[sy0:sy1, sx0:sx1]
        return out
def fit_loss(params, target):
        """Loss = green_uncovered_fraction + 0.5 * template_overshoot_fraction.

        Reaches 0 when template exactly covers the visible green and nothing more.
        """
        cx, cy, theta, scale = params
        if scale < 0.5 or scale > 2.5:
            return 10.0
        t = render_t(target.shape, cx, cy, theta, scale)
        inter = (t & target).sum()
        green_total = target.sum()
        tpl_total = t.sum()
        if green_total == 0 or tpl_total == 0:
            return 10.0
        uncovered = 1.0 - inter / green_total      # green pixels not under T
        overshoot = 1.0 - inter / tpl_total        # T pixels not on green
        return uncovered + 0.5 * overshoot
def fit_t_pose(frame, multi_start=True):
        """Optimize (x, y, theta, scale) by minimizing the coverage loss.

        Multi-start over angles to escape local minima, then Powell to convergence.
        """
        target = green_mask(frame)
        lab, n = label(target)
        if n == 0:
            return None
        sizes = [(lab == i).sum() for i in range(1, n + 1)]
        biggest = int(np.argmax(sizes)) + 1
        target = (lab == biggest)
        ys, xs = np.where(target)
        if len(xs) < 20:
            return None
        cx0, cy0 = float(xs.mean()), float(ys.mean())

        best = None
        angle_starts = range(0, 360, 30) if multi_start else [0]
        for theta0 in angle_starts:
            for scale0 in [1.0, 1.2]:
                x0 = np.array([cx0, cy0, float(theta0), scale0])
                try:
                    res = minimize(
                        fit_loss, x0, args=(target,),
                        method="Powell",
                        options={"xtol": 0.1, "ftol": 1e-4, "maxiter": 200},
                    )
                    if best is None or res.fun < best[0]:
                        cx, cy, theta, scale = res.x
                        best = (res.fun, cx, cy, theta % 360, scale)
                except Exception:
                    continue
        if best is None:
            return None
        loss, cx, cy, theta, scale = best
        # Convert loss to IoU-like score for reporting
        rendered = render_t(target.shape, cx, cy, theta, scale)
        inter = (rendered & target).sum()
        union = (rendered | target).sum()
        iou_val = float(inter) / float(union) if union > 0 else 0.0
        return (iou_val, cx, cy, theta, scale, loss)
def outline_full_t(frame, pose, color=(255, 0, 0), width=2):
        if pose is None:
            return frame.convert("RGB").copy()
        _, cx, cy, theta, scale, _ = pose
        H, W = np.array(frame).shape[:2]
        t_mask = render_t((H, W), cx, cy, theta, scale)
        edge = binary_dilation(t_mask) & ~t_mask
        if width > 1:
            edge = binary_dilation(edge, iterations=width - 1)
        arr = np.array(frame.convert("RGB")).copy()
        arr[edge] = color
        return Image.fromarray(arr)
ep_meta_path = hf_hub_download(
        repo_id="lerobot/pusht",
        filename="meta/episodes/chunk-000/file-000.parquet",
        repo_type="dataset",
    )
ep_meta = pq.read_table(ep_meta_path)
episode_starts = ep_meta.column("dataset_from_index").to_pylist()
print(f"{len(episode_starts)} episodes loaded")

In [ ]:
sample_episodes = [0, 25, 50, 100, 150, 200]
goal_panels = []
for ep_idx in sample_episodes:
        frame = decode_frame(pusht_mp4, episode_starts[ep_idx])
        pose = fit_t_pose(frame, multi_start=True)
        outlined = outline_full_t(frame, pose, color=(255, 0, 0), width=2)
        if pose is None:
            caption = f"ep {ep_idx}: no fit"
        else:
            iou_v, cx, cy, theta, scale, loss = pose
            caption = f"ep {ep_idx}: x={cx:.1f} y={cy:.1f} θ={theta:.0f}° s={scale:.2f} (IoU={iou_v:.2f}, loss={loss:.3f})"
        goal_panels.append(mo.vstack([
            mo.image(outlined.resize((192, 192), Image.NEAREST), width=192),
            mo.md(caption),
        ]))
mo.vstack([
        mo.md("### Full goal-T outline via Powell optimization (converged)"),
        mo.hstack(goal_panels, wrap=True, gap=1),
    ])

In [ ]:
import pathlib
import pandas as pd
GOAL_POSES_PATH = pathlib.Path("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
def fit_all_episodes():
        rows = []
        for ep_idx, start in enumerate(episode_starts):
            frame = decode_frame(pusht_mp4, start)
            pose = fit_t_pose(frame, multi_start=True)
            if pose is None:
                rows.append({"episode": ep_idx, "iou": None, "x": None, "y": None,
                             "theta_deg": None, "scale": None, "loss": None})
            else:
                iou_v, cx, cy, theta, scale, loss = pose
                rows.append({
                    "episode": ep_idx, "iou": float(iou_v),
                    "x": float(cx), "y": float(cy), "theta_deg": float(theta),
                    "scale": float(scale), "loss": float(loss),
                })
            if (ep_idx + 1) % 20 == 0:
                print(f"  {ep_idx + 1}/{len(episode_starts)}")
        return pd.DataFrame(rows)
if GOAL_POSES_PATH.exists():
        goal_poses_df = pd.read_parquet(GOAL_POSES_PATH)
        print(f"loaded cached fits from {GOAL_POSES_PATH} ({len(goal_poses_df)} rows)")
    else:
        print("fitting all 206 episodes (~1 min)...")
        goal_poses_df = fit_all_episodes()
        goal_poses_df.to_parquet(GOAL_POSES_PATH, index=False)
        print(f"saved to {GOAL_POSES_PATH}")
goal_poses_df

In [ ]:
import time as _time
import av as _av
from scipy.ndimage import label as _label
from scipy.optimize import minimize as _minimize
BLOCK_POSES_PATH = pathlib.Path("/Users/dimavremenko/Desktop/hackathon-pydata/block_poses.parquet")
def _gray_t_mask(frame):
        arr = np.array(frame)
        r, g, b = arr[..., 0].astype(int), arr[..., 1].astype(int), arr[..., 2].astype(int)
        m = (
            (g - r >= 8) & (g - r < 40) &
            (b - r >= 8) &
            (b - g >= -5) & (b - g < 30) &
            (r > 60) & (r < 200)
        )
        lab, n = _label(m)
        if n == 0:
            return None
        sizes = [(lab == i).sum() for i in range(1, n + 1)]
        biggest = int(np.argmax(sizes)) + 1
        return (lab == biggest)
def _fit_loss(params, target):
        cx, cy, theta, scale = params
        if scale < 0.5 or scale > 2.5:
            return 10.0
        t = render_t(target.shape, cx, cy, theta, scale)
        inter = (t & target).sum()
        g_total = target.sum()
        tpl_total = t.sum()
        if g_total == 0 or tpl_total == 0:
            return 10.0
        return (1 - inter / g_total) + 0.5 * (1 - inter / tpl_total)
def _fit_warm(target, warm):
        ys, xs = np.where(target)
        if len(xs) < 20:
            return None
        cx0, cy0 = float(xs.mean()), float(ys.mean())
        if warm is not None:
            x0 = np.array([warm[1], warm[2], warm[3], warm[4]])
            res = _minimize(_fit_loss, x0, args=(target,), method="Powell",
                            options={"xtol": 0.1, "ftol": 1e-3, "maxiter": 50})
            cx, cy, theta, scale = res.x
            loss = res.fun
        else:
            best = None
            for theta0 in range(0, 360, 30):
                x0 = np.array([cx0, cy0, float(theta0), 1.1])
                res = _minimize(_fit_loss, x0, args=(target,), method="Powell",
                                options={"xtol": 0.2, "ftol": 1e-2, "maxiter": 100})
                if best is None or res.fun < best[0]:
                    best = (res.fun, res.x)
            loss, x_ = best
            cx, cy, theta, scale = x_
        rendered = render_t(target.shape, cx, cy, theta, scale)
        inter = (rendered & target).sum()
        union = (rendered | target).sum()
        iou_v = float(inter) / float(union) if union else 0.0
        return (iou_v, float(cx), float(cy), float(theta % 360), float(scale), float(loss))
def track_all():
        ep_idx_per_frame = []
        for ep_idx, (start_i, length_i) in enumerate(
            zip(ep_meta.column("dataset_from_index").to_pylist(),
                ep_meta.column("length").to_pylist())
        ):
            ep_idx_per_frame.extend([ep_idx] * length_i)
        total = len(ep_idx_per_frame)

        container = _av.open(pusht_mp4)
        stream = container.streams.video[0]

        rows = []
        prev_pose = None
        prev_ep = -1
        t0 = _time.time()
        for i, frame in enumerate(container.decode(stream)):
            if i >= total:
                break
            ep = ep_idx_per_frame[i]
            if ep != prev_ep:
                prev_pose = None
                prev_ep = ep
            pil = frame.to_image()
            target = _gray_t_mask(pil)
            if target is None:
                pose = None
            else:
                pose = _fit_warm(target, prev_pose)
            if pose is not None:
                prev_pose = pose
                iou_v, cx, cy, theta, scale, loss = pose
                rows.append({"episode": ep, "frame_global": i, "x": cx, "y": cy,
                             "theta_deg": theta, "scale": scale, "iou": iou_v, "loss": loss})
            else:
                rows.append({"episode": ep, "frame_global": i, "x": None, "y": None,
                             "theta_deg": None, "scale": None, "iou": None, "loss": None})
            if (i + 1) % 2000 == 0:
                elapsed = _time.time() - t0
                rate = (i + 1) / elapsed
                eta = (total - i - 1) / rate
                print(f"  {i+1}/{total} frames ({rate:.0f}/s, ETA {eta:.0f}s)")
        container.close()
        print(f"done in {_time.time() - t0:.1f}s")
        return pd.DataFrame(rows)
if BLOCK_POSES_PATH.exists():
        block_poses_df = pd.read_parquet(BLOCK_POSES_PATH)
        print(f"loaded cached fits from {BLOCK_POSES_PATH} ({len(block_poses_df)} rows)")
    else:
        print("tracking all 25,650 frames (~3 min)...")
        block_poses_df = track_all()
        block_poses_df.to_parquet(BLOCK_POSES_PATH, index=False)
        print(f"saved to {BLOCK_POSES_PATH}")
block_poses_df.head()

In [ ]:
import pathlib as _pl
DATA_PARQUET_PATH = _pl.Path("/Users/dimavremenko/Desktop/hackathon-pydata/training_dataset.parquet")
ENV_TO_PIXEL = 0.185
ACTION_CHUNK = 8
def build_training_dataset():
        raw = pq.read_table(hf_hub_download(
            repo_id="lerobot/pusht",
            filename="data/chunk-000/file-000.parquet",
            repo_type="dataset",
        ))
        state = np.array(raw.column("observation.state").to_pylist()) * ENV_TO_PIXEL
        action = np.array(raw.column("action").to_pylist()) * ENV_TO_PIXEL
        ep_idx = np.array(raw.column("episode_index").to_pylist())
        frame_idx = np.array(raw.column("frame_index").to_pylist())
        reward = np.array(raw.column("next.reward").to_pylist())

        blk = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/block_poses.parquet")
        blk = blk.sort_values("frame_global").reset_index(drop=True)
        block_xy = blk[["x", "y"]].to_numpy()
        block_theta = blk["theta_deg"].to_numpy()

        goal = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
        goal_xy = goal[["x", "y"]].to_numpy()
        goal_theta = goal["theta_deg"].to_numpy()
        goal_per_frame_xy = goal_xy[ep_idx]
        goal_per_frame_theta = goal_theta[ep_idx]

        chunks = np.zeros((len(action), ACTION_CHUNK, 2), dtype=np.float32)
        for i in range(len(action)):
            ep_end = i
            while ep_end + 1 < len(action) and ep_idx[ep_end + 1] == ep_idx[i]:
                ep_end += 1
            for k in range(ACTION_CHUNK):
                src = min(i + k, ep_end)
                chunks[i, k] = action[src]

        def sincos(theta_deg):
            rad = np.deg2rad(theta_deg)
            return np.stack([np.sin(rad), np.cos(rad)], axis=-1)
        blk_sc = sincos(block_theta)
        goal_sc = sincos(goal_per_frame_theta)

        X = np.concatenate([state, block_xy, blk_sc, goal_per_frame_xy, goal_sc], axis=1).astype(np.float32)
        Y = chunks.reshape(len(chunks), -1)

        out = pd.DataFrame({"episode": ep_idx, "frame": frame_idx, "reward": reward})
        for i in range(X.shape[1]):
            out[f"x{i}"] = X[:, i]
        for i in range(Y.shape[1]):
            out[f"y{i}"] = Y[:, i]
        return out
if DATA_PARQUET_PATH.exists():
        training_df = pd.read_parquet(DATA_PARQUET_PATH)
        print(f"loaded cached training table ({len(training_df)} rows)")
    else:
        print("building training dataset...")
        training_df = build_training_dataset()
        training_df.to_parquet(DATA_PARQUET_PATH, index=False)
        print(f"saved: {len(training_df)} rows")
X_COLS = [c for c in training_df.columns if c.startswith("x")]
Y_COLS = [c for c in training_df.columns if c.startswith("y")]
print(f"X={len(X_COLS)}D, Y={len(Y_COLS)}D (= {ACTION_CHUNK} actions x 2)")
training_df.head()

In [ ]:
_rng = np.random.RandomState(42)
_all_eps = sorted(training_df.episode.unique())
_rng.shuffle(_all_eps)
_split_at = int(0.8 * len(_all_eps))
train_eps = set(_all_eps[:_split_at])
val_eps = set(_all_eps[_split_at:])
_tm = training_df.episode.isin(train_eps)
_vm = training_df.episode.isin(val_eps)
X_train = training_df.loc[_tm, X_COLS].to_numpy().astype(np.float32)
Y_train = training_df.loc[_tm, Y_COLS].to_numpy().astype(np.float32)
X_val = training_df.loc[_vm, X_COLS].to_numpy().astype(np.float32)
Y_val = training_df.loc[_vm, Y_COLS].to_numpy().astype(np.float32)
X_mean = X_train.mean(0)
X_std = X_train.std(0) + 1e-6
Y_mean = Y_train.mean(0)
Y_std = Y_train.std(0) + 1e-6
X_train_n = (X_train - X_mean) / X_std
Y_train_n = (Y_train - Y_mean) / Y_std
X_val_n = (X_val - X_mean) / X_std
Y_val_n = (Y_val - Y_mean) / Y_std
print(f"train episodes: {len(train_eps)} ({len(X_train)} frames)")
print(f"val   episodes: {len(val_eps)} ({len(X_val)} frames)")

In [ ]:
import torch
import torch.nn as nn
class PushTPolicy(nn.Module):
        def __init__(self, in_dim, out_dim, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim),
            )
        def forward(self, x):
            return self.net(x)
torch.manual_seed(0)
nn_device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"using device: {nn_device}")
model = PushTPolicy(X_train.shape[1], Y_train.shape[1]).to(nn_device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
X_train_t = torch.tensor(X_train_n).to(nn_device)
Y_train_t = torch.tensor(Y_train_n).to(nn_device)
X_val_t = torch.tensor(X_val_n).to(nn_device)
Y_val_t = torch.tensor(Y_val_n).to(nn_device)
EPOCHS = 40
BATCH = 512
history_rows = []
import time as _t
t0 = _t.time()
for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(X_train_t), device=nn_device)
        batch_losses = []
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i+BATCH]
            pred = model(X_train_t[idx])
            batch_loss = ((pred - Y_train_t[idx]) ** 2).mean()
            opt.zero_grad(); batch_loss.backward(); opt.step()
            batch_losses.append(batch_loss.item())
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_mse = ((val_pred - Y_val_t) ** 2).mean().item()
            val_pred_dn = val_pred.cpu().numpy() * Y_std + Y_mean
            val_rmse_px = float(np.sqrt(((val_pred_dn - Y_val) ** 2).mean()))
        tr = float(np.mean(batch_losses))
        history_rows.append({"epoch": epoch + 1, "train_mse": tr, "val_mse": val_mse, "val_rmse_px": val_rmse_px})
        print(f"epoch {epoch+1:3d}: train MSE={tr:.4f}, val MSE={val_mse:.4f}, val RMSE={val_rmse_px:.2f} px")
print(f"\ntrain time: {_t.time()-t0:.1f}s on {nn_device}")
history_df = pd.DataFrame(history_rows)
history_df.tail()

In [ ]:
DATA_V2_PATH = pathlib.Path("/Users/dimavremenko/Desktop/hackathon-pydata/training_v2.parquet")
HISTORY_K = 4
def build_v2():
        raw = pq.read_table(hf_hub_download(
            repo_id="lerobot/pusht",
            filename="data/chunk-000/file-000.parquet",
            repo_type="dataset",
        ))
        state = np.array(raw.column("observation.state").to_pylist()) * ENV_TO_PIXEL
        action = np.array(raw.column("action").to_pylist()) * ENV_TO_PIXEL
        ep_idx = np.array(raw.column("episode_index").to_pylist())
        frame_idx = np.array(raw.column("frame_index").to_pylist())

        blk = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/block_poses.parquet")
        blk = blk.sort_values("frame_global").reset_index(drop=True)
        block_xy = blk[["x", "y"]].to_numpy()
        block_theta_rad = np.deg2rad(blk["theta_deg"].to_numpy())

        goal = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
        goal_xy = goal[["x", "y"]].to_numpy()
        goal_theta_rad = np.deg2rad(goal["theta_deg"].to_numpy())
        goal_per_frame_xy = goal_xy[ep_idx]
        goal_per_frame_theta = goal_theta_rad[ep_idx]

        pusher_rel = state - goal_per_frame_xy
        block_rel = block_xy - goal_per_frame_xy
        block_theta_rel = block_theta_rad - goal_per_frame_theta

        per_frame = np.concatenate([
            pusher_rel,
            block_rel,
            np.sin(block_theta_rel)[:, None],
            np.cos(block_theta_rel)[:, None],
        ], axis=1).astype(np.float32)

        N = len(per_frame)
        X = np.zeros((N, HISTORY_K * per_frame.shape[1]), dtype=np.float32)
        for i in range(N):
            cur_ep = ep_idx[i]
            for k in range(HISTORY_K):
                src = i - (HISTORY_K - 1 - k)
                if src < 0 or ep_idx[src] != cur_ep:
                    # earliest frame of this episode
                    src = i
                    while src - 1 >= 0 and ep_idx[src - 1] == cur_ep:
                        src -= 1
                X[i, k*per_frame.shape[1]:(k+1)*per_frame.shape[1]] = per_frame[src]

        chunks = np.zeros((N, ACTION_CHUNK, 2), dtype=np.float32)
        for i in range(N):
            ep_end = i
            while ep_end + 1 < N and ep_idx[ep_end + 1] == ep_idx[i]:
                ep_end += 1
            for k in range(ACTION_CHUNK):
                chunks[i, k] = action[min(i + k, ep_end)]
        Y = chunks.reshape(N, -1)

        out = pd.DataFrame({"episode": ep_idx, "frame": frame_idx})
        for i in range(X.shape[1]):
            out[f"x{i}"] = X[:, i]
        for i in range(Y.shape[1]):
            out[f"y{i}"] = Y[:, i]
        return out
if DATA_V2_PATH.exists():
        training_v2 = pd.read_parquet(DATA_V2_PATH)
        print(f"loaded cached v2 dataset ({len(training_v2)} rows)")
    else:
        print("building v2 dataset (history + goal-relative)...")
        training_v2 = build_v2()
        training_v2.to_parquet(DATA_V2_PATH, index=False)
        print(f"saved {len(training_v2)} rows")
X2_COLS = [c for c in training_v2.columns if c.startswith("x")]
Y2_COLS = [c for c in training_v2.columns if c.startswith("y")]
print(f"X={len(X2_COLS)}D ({HISTORY_K} frames x 6 features), Y={len(Y2_COLS)}D")
training_v2.head()

In [ ]:
_rng2 = np.random.RandomState(42)
_all_eps2 = sorted(training_v2.episode.unique())
_rng2.shuffle(_all_eps2)
_split_at2 = int(0.8 * len(_all_eps2))
train_eps_v2 = set(_all_eps2[:_split_at2])
val_eps_v2 = set(_all_eps2[_split_at2:])
_tm2 = training_v2.episode.isin(train_eps_v2)
_vm2 = training_v2.episode.isin(val_eps_v2)
X_train_v2 = training_v2.loc[_tm2, X2_COLS].to_numpy().astype(np.float32)
Y_train_v2 = training_v2.loc[_tm2, Y2_COLS].to_numpy().astype(np.float32)
X_val_v2 = training_v2.loc[_vm2, X2_COLS].to_numpy().astype(np.float32)
Y_val_v2 = training_v2.loc[_vm2, Y2_COLS].to_numpy().astype(np.float32)
X_mean_v2 = X_train_v2.mean(0)
X_std_v2 = X_train_v2.std(0) + 1e-6
Y_mean_v2 = Y_train_v2.mean(0)
Y_std_v2 = Y_train_v2.std(0) + 1e-6
X_train_v2_n = (X_train_v2 - X_mean_v2) / X_std_v2
Y_train_v2_n = (Y_train_v2 - Y_mean_v2) / Y_std_v2
X_val_v2_n = (X_val_v2 - X_mean_v2) / X_std_v2
Y_val_v2_n = (Y_val_v2 - Y_mean_v2) / Y_std_v2
print(f"train: {len(X_train_v2)} frames ({len(train_eps_v2)} eps)")
print(f"val:   {len(X_val_v2)} frames ({len(val_eps_v2)} eps)")
print(f"X={X_train_v2.shape[1]}D, Y={Y_train_v2.shape[1]}D")

In [ ]:
import torch as _torch_v2
from torch import nn as _nn_v2
class PushTPolicyV2(_nn_v2.Module):
        def __init__(self, in_dim, out_dim, hidden=512, depth=5, dropout=0.1):
            super().__init__()
            layers = [_nn_v2.Linear(in_dim, hidden), _nn_v2.GELU(), _nn_v2.Dropout(dropout)]
            for _ in range(depth - 1):
                layers += [_nn_v2.Linear(hidden, hidden), _nn_v2.GELU(), _nn_v2.Dropout(dropout)]
            layers += [_nn_v2.Linear(hidden, out_dim)]
            self.net = _nn_v2.Sequential(*layers)
        def forward(self, x):
            return self.net(x)
def _train_v2():
        import time as _time_v2
        import copy as _copy_v2

        _torch_v2.manual_seed(0)
        device = "mps" if _torch_v2.backends.mps.is_available() else "cpu"
        print(f"device: {device}")

        model = PushTPolicyV2(X_train_v2.shape[1], Y_train_v2.shape[1], hidden=512, depth=5, dropout=0.1).to(device)
        n_params = sum(p.numel() for p in model.parameters())
        print(f"params: {n_params:,}")

        EPOCHS = 80
        BATCH = 1024
        opt = _torch_v2.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sched = _torch_v2.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

        Xtr = _torch_v2.tensor(X_train_v2_n).to(device)
        Ytr = _torch_v2.tensor(Y_train_v2_n).to(device)
        Xv = _torch_v2.tensor(X_val_v2_n).to(device)
        Yv = _torch_v2.tensor(Y_val_v2_n).to(device)

        history = []
        best_val = float("inf")
        best_state = None
        patience, since_best = 15, 0
        t0 = _time_v2.time()
        for ep in range(EPOCHS):
            model.train()
            perm = _torch_v2.randperm(len(Xtr), device=device)
            losses_ep = []
            for i in range(0, len(perm), BATCH):
                idx = perm[i:i+BATCH]
                pred = model(Xtr[idx])
                L = ((pred - Ytr[idx]) ** 2).mean()
                opt.zero_grad(); L.backward(); opt.step()
                losses_ep.append(L.item())
            sched.step()
            model.eval()
            with _torch_v2.no_grad():
                vp = model(Xv)
                vmse = ((vp - Yv) ** 2).mean().item()
                vp_dn = vp.cpu().numpy() * Y_std_v2 + Y_mean_v2
                vrmse = float(np.sqrt(((vp_dn - Y_val_v2) ** 2).mean()))
            tr = float(np.mean(losses_ep))
            history.append({"epoch": ep + 1, "train_mse": tr, "val_mse": vmse, "val_rmse_px": vrmse, "lr": opt.param_groups[0]["lr"]})
            marker = ""
            if vmse < best_val:
                best_val = vmse
                best_state = _copy_v2.deepcopy(model.state_dict())
                since_best = 0
                marker = "  ★"
            else:
                since_best += 1
            if ep < 5 or ep % 2 == 0 or marker:
                print(f"ep {ep+1:3d}: train MSE={tr:.4f}  val MSE={vmse:.4f}  val RMSE={vrmse:.2f}px{marker}")
            if since_best >= patience:
                print(f"early stop at epoch {ep+1}")
                break

        model.load_state_dict(best_state)
        print(f"\ntrain time: {_time_v2.time()-t0:.1f}s on {device}")
        print(f"best val MSE: {best_val:.4f}, best val RMSE: {min(h['val_rmse_px'] for h in history):.2f}px")
        return model, history, device
model_v2, _history_list, nn_device_v2 = _train_v2()
history_v2_df = pd.DataFrame(_history_list)
history_v2_df.tail(10)

In [ ]:
import sys as _sys
for _m in [_m for _m in list(_sys.modules) if _m.startswith(("pymunk", "gym_pusht"))]:
        del _sys.modules[_m]
import gymnasium as _gym
import gym_pusht as _gp
import torch as _torch_r
import pickle as _pickle
import collections as _collections
import time as _time_r
with open("/Users/dimavremenko/Desktop/hackathon-pydata/norm_v2.pkl", "rb") as _f:
        _norm = _pickle.load(_f)
X_mean_r = _norm["X_mean"]
X_std_r = _norm["X_std"]
Y_mean_r = _norm["Y_mean"]
Y_std_r = _norm["Y_std"]
HISTORY_K_r = _norm["history_k"]
ACTION_CHUNK_r = _norm["action_chunk"]
ENV_TO_PIXEL_r = _norm["env_to_pixel"]
ROLLOUT_DEVICE = "mps" if _torch_r.backends.mps.is_available() else "cpu"
def _make_features(history):
        feats = []
        for pusher, block, btheta, goal_xy, gtheta in history:
            pusher_px = pusher * ENV_TO_PIXEL_r
            block_px = block * ENV_TO_PIXEL_r
            goal_px = goal_xy * ENV_TO_PIXEL_r
            pusher_rel = pusher_px - goal_px
            block_rel = block_px - goal_px
            theta_rel = btheta - gtheta
            feats.extend([pusher_rel[0], pusher_rel[1],
                          block_rel[0], block_rel[1],
                          np.sin(theta_rel), np.cos(theta_rel)])
        return np.array(feats, dtype=np.float32)
def _predict_actions(model, history):
        feats = _make_features(history)
        feats_n = (feats - X_mean_r) / X_std_r
        with _torch_r.no_grad():
            pred = model(_torch_r.tensor(feats_n).unsqueeze(0).to(ROLLOUT_DEVICE))
        pred_dn = pred.cpu().numpy().reshape(-1) * Y_std_r + Y_mean_r
        actions_px = pred_dn.reshape(ACTION_CHUNK_r, 2)
        return actions_px / ENV_TO_PIXEL_r
def run_episode(model, seed, max_steps=300, execute_per_chunk=4):
        env = _gym.make("gym_pusht/PushT-v0", obs_type="state")
        obs, info = env.reset(seed=seed)
        goal_xy = info["goal_pose"][:2]
        goal_theta = info["goal_pose"][2]
        pusher = obs[:2].copy()
        block_xy = obs[2:4].copy()
        block_theta = obs[4]
        hist = _collections.deque(maxlen=HISTORY_K_r)
        for _ in range(HISTORY_K_r):
            hist.append((pusher.copy(), block_xy.copy(), float(block_theta),
                         goal_xy.copy(), float(goal_theta)))

        max_reward = 0.0
        success = False
        steps = 0
        while steps < max_steps:
            actions = _predict_actions(model, list(hist))
            for k in range(execute_per_chunk):
                if steps >= max_steps:
                    break
                act = np.clip(actions[k], 0.0, 512.0).astype(np.float32)
                obs, reward, term, trunc, info = env.step(act)
                steps += 1
                max_reward = max(max_reward, float(reward))
                if info.get("is_success", False):
                    success = True
                pusher = obs[:2].copy()
                block_xy = obs[2:4].copy()
                block_theta = obs[4]
                hist.append((pusher.copy(), block_xy.copy(), float(block_theta),
                             goal_xy.copy(), float(goal_theta)))
                if term or trunc or success:
                    break
            if term or trunc or success:
                break
        env.close()
        return success, max_reward, steps
def _smoke():
        t = _time_r.time()
        s, r, n = run_episode(model_v2, seed=0, max_steps=300, execute_per_chunk=4)
        print(f"smoke: success={s}, max_reward={r:.3f}, steps={n} ({_time_r.time()-t:.1f}s)")
_smoke()

In [ ]:
print("running 50 rollouts...")
_t0 = _time_r.time()
rollout_results = []
for _seed in range(50):
        _succ, _maxr, _steps = run_episode(model_v2, seed=_seed, max_steps=300, execute_per_chunk=4)
        rollout_results.append({"seed": _seed, "success": _succ, "max_reward": _maxr, "steps": _steps})
        if (_seed + 1) % 10 == 0:
            _so_far = pd.DataFrame(rollout_results)
            print(f"  {_seed+1}/50: success rate so far = {_so_far.success.mean()*100:.0f}%, "
                  f"mean max_reward = {_so_far.max_reward.mean():.2f}")
rollout_df = pd.DataFrame(rollout_results)
print(f"\nFINAL: success rate = {rollout_df.success.mean()*100:.0f}% ({rollout_df.success.sum()}/50)")
print(f"       mean max_reward = {rollout_df.max_reward.mean():.2f}")
print(f"       reward > 0.95: {(rollout_df.max_reward > 0.95).sum()}/50 ({(rollout_df.max_reward > 0.95).mean()*100:.0f}%)")
print(f"       reward > 0.50: {(rollout_df.max_reward > 0.50).sum()}/50 ({(rollout_df.max_reward > 0.50).mean()*100:.0f}%)")
print(f"       time: {_time_r.time() - _t0:.1f}s")
rollout_df

In [ ]:
DATA_V3_PATH = pathlib.Path("/Users/dimavremenko/Desktop/hackathon-pydata/training_v3.parquet")
HISTORY_K_v3 = 4
ACTION_CHUNK_v3 = 8
PIXEL_TO_ENV = 512.0 / 96.0
def build_v3():
        raw = pq.read_table(hf_hub_download(
            repo_id="lerobot/pusht",
            filename="data/chunk-000/file-000.parquet",
            repo_type="dataset",
        ))
        state = np.array(raw.column("observation.state").to_pylist())  # ENV coords (0-512)
        action = np.array(raw.column("action").to_pylist())             # ENV coords (0-512)
        ep_idx = np.array(raw.column("episode_index").to_pylist())
        frame_idx = np.array(raw.column("frame_index").to_pylist())

        blk = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/block_poses.parquet")
        blk = blk.sort_values("frame_global").reset_index(drop=True)
        block_xy_env = blk[["x", "y"]].to_numpy() * PIXEL_TO_ENV  # convert tracker px -> env
        block_theta_rad = np.deg2rad(blk["theta_deg"].to_numpy())

        goal_df = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
        # Use per-episode tracker goal but in env coords
        goal_xy_env = goal_df[["x", "y"]].to_numpy() * PIXEL_TO_ENV
        goal_theta_rad = np.deg2rad(goal_df["theta_deg"].to_numpy())
        goal_per_frame_xy = goal_xy_env[ep_idx]
        goal_per_frame_theta = goal_theta_rad[ep_idx]

        # Features: goal-relative (everything subtract goal in env coords)
        pusher_rel = state - goal_per_frame_xy
        block_rel = block_xy_env - goal_per_frame_xy
        block_theta_rel = block_theta_rad - goal_per_frame_theta

        per_frame = np.concatenate([
            pusher_rel,
            block_rel,
            np.sin(block_theta_rel)[:, None],
            np.cos(block_theta_rel)[:, None],
        ], axis=1).astype(np.float32)

        N = len(per_frame)
        X = np.zeros((N, HISTORY_K_v3 * per_frame.shape[1]), dtype=np.float32)
        for i in range(N):
            cur_ep = ep_idx[i]
            for k in range(HISTORY_K_v3):
                src = i - (HISTORY_K_v3 - 1 - k)
                if src < 0 or ep_idx[src] != cur_ep:
                    src = i
                    while src - 1 >= 0 and ep_idx[src - 1] == cur_ep:
                        src -= 1
                X[i, k*per_frame.shape[1]:(k+1)*per_frame.shape[1]] = per_frame[src]

        # Action chunks: goal-relative actions (so model output is "where to go relative to goal")
        chunks = np.zeros((N, ACTION_CHUNK_v3, 2), dtype=np.float32)
        for i in range(N):
            ep_end = i
            while ep_end + 1 < N and ep_idx[ep_end + 1] == ep_idx[i]:
                ep_end += 1
            for k in range(ACTION_CHUNK_v3):
                src = min(i + k, ep_end)
                chunks[i, k] = action[src] - goal_per_frame_xy[i]  # goal-relative action
        Y = chunks.reshape(N, -1)

        out = pd.DataFrame({"episode": ep_idx, "frame": frame_idx})
        for i in range(X.shape[1]):
            out[f"x{i}"] = X[:, i]
        for i in range(Y.shape[1]):
            out[f"y{i}"] = Y[:, i]
        return out
if DATA_V3_PATH.exists():
        training_v3 = pd.read_parquet(DATA_V3_PATH)
        print(f"loaded cached v3 dataset ({len(training_v3)} rows)")
    else:
        print("building v3 dataset (env coords, goal-relative everything)...")
        training_v3 = build_v3()
        training_v3.to_parquet(DATA_V3_PATH, index=False)
        print(f"saved {len(training_v3)} rows")
X3_COLS = [c for c in training_v3.columns if c.startswith("x")]
Y3_COLS = [c for c in training_v3.columns if c.startswith("y")]
print(f"X={len(X3_COLS)}D, Y={len(Y3_COLS)}D")
training_v3.head()

In [ ]:
_r3 = np.random.RandomState(42)
_eps3 = sorted(training_v3.episode.unique())
_r3.shuffle(_eps3)
_split3 = int(0.8 * len(_eps3))
train_eps_v3 = set(_eps3[:_split3])
val_eps_v3 = set(_eps3[_split3:])
_tm3 = training_v3.episode.isin(train_eps_v3)
_vm3 = training_v3.episode.isin(val_eps_v3)
X_train_v3 = training_v3.loc[_tm3, X3_COLS].to_numpy().astype(np.float32)
Y_train_v3 = training_v3.loc[_tm3, Y3_COLS].to_numpy().astype(np.float32)
X_val_v3 = training_v3.loc[_vm3, X3_COLS].to_numpy().astype(np.float32)
Y_val_v3 = training_v3.loc[_vm3, Y3_COLS].to_numpy().astype(np.float32)
X_mean_v3 = X_train_v3.mean(0)
X_std_v3 = X_train_v3.std(0) + 1e-6
Y_mean_v3 = Y_train_v3.mean(0)
Y_std_v3 = Y_train_v3.std(0) + 1e-6
X_train_v3_n = (X_train_v3 - X_mean_v3) / X_std_v3
Y_train_v3_n = (Y_train_v3 - Y_mean_v3) / Y_std_v3
X_val_v3_n = (X_val_v3 - X_mean_v3) / X_std_v3
Y_val_v3_n = (Y_val_v3 - Y_mean_v3) / Y_std_v3
print(f"train: {len(X_train_v3)} frames ({len(train_eps_v3)} eps)")
print(f"val:   {len(X_val_v3)} frames ({len(val_eps_v3)} eps)")
print(f"X={X_train_v3.shape[1]}D, Y={Y_train_v3.shape[1]}D")
print(f"Y range (goal-rel action): [{Y_train_v3.min():.0f}, {Y_train_v3.max():.0f}]")

In [ ]:
import torch as _torch_v3
from torch import nn as _nn_v3
class PolicyV3(_nn_v3.Module):
        def __init__(self, in_dim, out_dim, hidden=512, depth=5, dropout=0.1):
            super().__init__()
            layers = [_nn_v3.Linear(in_dim, hidden), _nn_v3.GELU(), _nn_v3.Dropout(dropout)]
            for _ in range(depth - 1):
                layers += [_nn_v3.Linear(hidden, hidden), _nn_v3.GELU(), _nn_v3.Dropout(dropout)]
            layers += [_nn_v3.Linear(hidden, out_dim)]
            self.net = _nn_v3.Sequential(*layers)
        def forward(self, x):
            return self.net(x)
def _train_v3():
        import time as _tt
        import copy as _cp

        _torch_v3.manual_seed(0)
        dev = "mps" if _torch_v3.backends.mps.is_available() else "cpu"
        print(f"device: {dev}")

        m = PolicyV3(X_train_v3.shape[1], Y_train_v3.shape[1]).to(dev)
        nparams = sum(p.numel() for p in m.parameters())
        print(f"params: {nparams:,}")

        EP = 80
        BS = 1024
        opt = _torch_v3.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
        sch = _torch_v3.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)

        Xtr = _torch_v3.tensor(X_train_v3_n).to(dev)
        Ytr = _torch_v3.tensor(Y_train_v3_n).to(dev)
        Xv = _torch_v3.tensor(X_val_v3_n).to(dev)
        Yv = _torch_v3.tensor(Y_val_v3_n).to(dev)

        hist = []
        best_val = float("inf"); best_state = None
        patience = 15; since_best = 0
        t0 = _tt.time()
        for ep in range(EP):
            m.train()
            perm = _torch_v3.randperm(len(Xtr), device=dev)
            ls = []
            for i in range(0, len(perm), BS):
                idx = perm[i:i+BS]
                p = m(Xtr[idx])
                L = ((p - Ytr[idx]) ** 2).mean()
                opt.zero_grad(); L.backward(); opt.step()
                ls.append(L.item())
            sch.step()
            m.eval()
            with _torch_v3.no_grad():
                vp = m(Xv)
                vmse = ((vp - Yv) ** 2).mean().item()
                vp_dn = vp.cpu().numpy() * Y_std_v3 + Y_mean_v3
                vrmse_env = float(np.sqrt(((vp_dn - Y_val_v3) ** 2).mean()))
                vrmse_px = vrmse_env * 96.0 / 512.0
            tr = float(np.mean(ls))
            hist.append({"epoch": ep+1, "train_mse": tr, "val_mse": vmse, "val_rmse_env": vrmse_env, "val_rmse_px": vrmse_px})
            marker = ""
            if vmse < best_val:
                best_val = vmse
                best_state = _cp.deepcopy(m.state_dict())
                since_best = 0
                marker = "  ★"
            else:
                since_best += 1
            if ep < 5 or ep % 2 == 0 or marker:
                print(f"ep {ep+1:3d}: tr MSE={tr:.4f}  v MSE={vmse:.4f}  v RMSE={vrmse_env:.1f} env ({vrmse_px:.2f} px){marker}")
            if since_best >= patience:
                print(f"early stop at {ep+1}")
                break
        m.load_state_dict(best_state)
        print(f"\ntrain time: {_tt.time()-t0:.1f}s, best val RMSE: {min(h['val_rmse_env'] for h in hist):.1f} env / {min(h['val_rmse_px'] for h in hist):.2f} px")
        # Save weights + normalizations
        import pickle as _pk
        _torch_v3.save({
            "state_dict": m.state_dict(),
            "in_dim": X_train_v3.shape[1], "out_dim": Y_train_v3.shape[1],
        }, "/Users/dimavremenko/Desktop/hackathon-pydata/model_v3.pt")
        with open("/Users/dimavremenko/Desktop/hackathon-pydata/norm_v3.pkl", "wb") as _f:
            _pk.dump({
                "X_mean": X_mean_v3, "X_std": X_std_v3,
                "Y_mean": Y_mean_v3, "Y_std": Y_std_v3,
                "history_k": 4, "action_chunk": 8,
                "pixel_to_env": 512.0 / 96.0,
            }, _f)
        return m, hist, dev
model_v3, _history_v3, nn_device_v3 = _train_v3()
history_v3_df = pd.DataFrame(_history_v3)
history_v3_df.tail(8)

In [ ]:
import sys as _sysv3
for _m in [_m for _m in list(_sysv3.modules) if _m.startswith(("pymunk", "gym_pusht"))]:
        del _sysv3.modules[_m]
import gymnasium as _gymv3
import gym_pusht as _gpv3
import collections as _colv3
import time as _timev3
import torch as _torch_rollout
import pandas as _pdv3
_goal_df = _pdv3.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
ROLLOUT_GOAL_ENV = (_goal_df.x.median() * 512.0/96.0, _goal_df.y.median() * 512.0/96.0)
ROLLOUT_GOAL_THETA = np.deg2rad(_goal_df.theta_deg.median())
print(f"rollout goal (env): {ROLLOUT_GOAL_ENV}, theta={ROLLOUT_GOAL_THETA:.3f}")
def _build_feats_env(history, goal_xy, goal_theta):
        feats = []
        for pusher, block, btheta in history:
            pr = pusher - goal_xy
            br = block - goal_xy
            tr = btheta - goal_theta
            feats.extend([pr[0], pr[1], br[0], br[1], np.sin(tr), np.cos(tr)])
        return np.array(feats, dtype=np.float32)
def _predict_actions_v3(model, history, goal_xy, goal_theta):
        feats = _build_feats_env(history, goal_xy, goal_theta)
        feats_n = (feats - X_mean_v3) / X_std_v3
        with _torch_rollout.no_grad():
            pred = model(_torch_rollout.tensor(feats_n).unsqueeze(0).to(nn_device_v3))
        pred_dn = pred.cpu().numpy().reshape(-1) * Y_std_v3 + Y_mean_v3
        actions_rel = pred_dn.reshape(8, 2)
        actions_abs = actions_rel + goal_xy
        return actions_abs
def run_episode_v3(model, seed, max_steps=300, execute_per_chunk=4, use_env_goal=False):
        env = _gymv3.make("gym_pusht/PushT-v0", obs_type="state")
        obs, info = env.reset(seed=seed)
        if use_env_goal:
            goal_xy = info["goal_pose"][:2]
            goal_theta = info["goal_pose"][2]
        else:
            goal_xy = np.array(ROLLOUT_GOAL_ENV)
            goal_theta = ROLLOUT_GOAL_THETA
        pusher = obs[:2].copy(); block_xy = obs[2:4].copy(); block_theta = obs[4]
        hist = _colv3.deque(maxlen=4)
        for _ in range(4):
            hist.append((pusher.copy(), block_xy.copy(), float(block_theta)))
        max_reward = 0.0; max_coverage = 0.0; success = False; steps = 0
        while steps < max_steps:
            actions = _predict_actions_v3(model, list(hist), goal_xy, goal_theta)
            for k in range(execute_per_chunk):
                if steps >= max_steps: break
                act = np.clip(actions[k], 0.0, 512.0).astype(np.float32)
                obs, reward, term, trunc, info = env.step(act)
                steps += 1
                max_reward = max(max_reward, float(reward))
                max_coverage = max(max_coverage, float(info.get("coverage", reward)))
                if info.get("is_success", False):
                    success = True
                pusher = obs[:2]; block_xy = obs[2:4]; block_theta = obs[4]
                hist.append((pusher.copy(), block_xy.copy(), float(block_theta)))
                if term or trunc or success: break
            if term or trunc or success: break
        env.close()
        return success, max_reward, max_coverage, steps
_t = _timev3.time()
_s, _r, _c, _n = run_episode_v3(model_v3, seed=0, execute_per_chunk=4, use_env_goal=False)
print(f"smoke (trained goal): success={_s}, max_r={_r:.3f}, max_cov={_c:.3f}, steps={_n} ({_timev3.time()-_t:.1f}s)")
_t = _timev3.time()
_s, _r, _c, _n = run_episode_v3(model_v3, seed=0, execute_per_chunk=4, use_env_goal=True)
print(f"smoke (env goal):     success={_s}, max_r={_r:.3f}, max_cov={_c:.3f}, steps={_n} ({_timev3.time()-_t:.1f}s)")

In [ ]:
import time as _te3
print("running 50 rollouts (trained goal)...")
_t0 = _te3.time()
_results = []
for _seed in range(50):
        _s, _r, _c, _n = run_episode_v3(model_v3, seed=_seed, max_steps=300, execute_per_chunk=4, use_env_goal=False)
        _results.append({"seed": _seed, "success": _s, "max_reward": _r, "max_coverage": _c, "steps": _n})
        if (_seed + 1) % 10 == 0:
            _df = pd.DataFrame(_results)
            print(f"  {_seed+1}/50: success={_df.success.mean()*100:.0f}%, "
                  f"mean max_r={_df.max_reward.mean():.2f}, mean max_cov={_df.max_coverage.mean():.2f}")
rollout_v3_df = pd.DataFrame(_results)
print(f"\nFINAL: success rate = {rollout_v3_df.success.mean()*100:.0f}% ({rollout_v3_df.success.sum()}/50)")
print(f"       mean max_reward  = {rollout_v3_df.max_reward.mean():.2f}")
print(f"       mean max_coverage = {rollout_v3_df.max_coverage.mean():.2f}")
print(f"       coverage > 0.95: {(rollout_v3_df.max_coverage > 0.95).sum()}/50")
print(f"       coverage > 0.50: {(rollout_v3_df.max_coverage > 0.50).sum()}/50")
print(f"       coverage > 0.40: {(rollout_v3_df.max_coverage > 0.40).sum()}/50")
print(f"       time: {_te3.time() - _t0:.1f}s")
rollout_v3_df.head(10)

In [ ]:
DATA_V4_PATH = pathlib.Path("/Users/dimavremenko/Desktop/hackathon-pydata/training_v4.parquet")
HISTORY_K_v4 = 4
ACTION_CHUNK_v4 = 8
PIXEL_TO_ENV_v4 = 512.0 / 96.0
T_KEYPOINTS_PX = np.array([
        [0.0, 0.0],
        [-11.34, -5.77],
        [12.66, -5.77],
        [0.66, 14.23],
    ], dtype=np.float32)
T_KEYPOINTS_ENV = T_KEYPOINTS_PX * PIXEL_TO_ENV_v4
def keypoints_at_pose(cx, cy, theta_rad):
        """Transform canonical keypoints to world coords at given pose (env units)."""
        cos_t, sin_t = np.cos(theta_rad), np.sin(theta_rad)
        R = np.array([[cos_t, -sin_t], [sin_t, cos_t]], dtype=np.float32)
        transformed = T_KEYPOINTS_ENV @ R.T
        transformed[:, 0] += cx
        transformed[:, 1] += cy
        return transformed
def build_v4():
        raw = pq.read_table(hf_hub_download(
            repo_id="lerobot/pusht",
            filename="data/chunk-000/file-000.parquet",
            repo_type="dataset",
        ))
        state = np.array(raw.column("observation.state").to_pylist())  # ENV (0-512)
        action = np.array(raw.column("action").to_pylist())             # ENV (0-512)
        ep_idx = np.array(raw.column("episode_index").to_pylist())
        frame_idx = np.array(raw.column("frame_index").to_pylist())

        blk = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/block_poses.parquet")
        blk = blk.sort_values("frame_global").reset_index(drop=True)
        block_xy_env = blk[["x", "y"]].to_numpy() * PIXEL_TO_ENV_v4
        block_theta_rad = np.deg2rad(blk["theta_deg"].to_numpy())

        goal_df = pd.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
        goal_xy_env = goal_df[["x", "y"]].to_numpy() * PIXEL_TO_ENV_v4
        goal_theta_rad = np.deg2rad(goal_df["theta_deg"].to_numpy())
        goal_pf_xy = goal_xy_env[ep_idx]
        goal_pf_theta = goal_theta_rad[ep_idx]

        N = len(state)
        # Per-frame features:
        # pusher_xy (2) + block_kps (4*2=8) + goal_kps (4*2=8)
        #   + (pusher -> each block_kp) vectors (4*2=8)
        #   + (block_kp -> goal_kp) vectors (4*2=8)
        # = 2 + 8 + 8 + 8 + 8 = 34 per frame
        # Goal-relative: subtract goal CENTER from positional features (vectors are translation invariant already)
        per_frame_dim = 34
        per_frame = np.zeros((N, per_frame_dim), dtype=np.float32)

        for i in range(N):
            gxy = goal_pf_xy[i]
            gth = goal_pf_theta[i]
            bxy = block_xy_env[i]
            bth = block_theta_rad[i]
            pusher = state[i]

            block_kps = keypoints_at_pose(bxy[0], bxy[1], bth)  # (4, 2)
            goal_kps = keypoints_at_pose(gxy[0], gxy[1], gth)   # (4, 2)

            # Goal-relative positions (translate so goal center = origin)
            pusher_rel = pusher - gxy            # (2,)
            block_kps_rel = block_kps - gxy      # (4, 2)
            goal_kps_rel = goal_kps - gxy        # (4, 2)

            # Vectors: pusher -> each block keypoint
            pusher_to_block = block_kps - pusher  # (4, 2)
            # Vectors: each block keypoint -> matching goal keypoint
            block_to_goal = goal_kps - block_kps  # (4, 2)

            feats = np.concatenate([
                pusher_rel.flatten(),         # 2
                block_kps_rel.flatten(),       # 8
                goal_kps_rel.flatten(),        # 8
                pusher_to_block.flatten(),     # 8
                block_to_goal.flatten(),       # 8
            ])
            per_frame[i] = feats

        # History stacking
        X = np.zeros((N, HISTORY_K_v4 * per_frame_dim), dtype=np.float32)
        for i in range(N):
            cur_ep = ep_idx[i]
            for k in range(HISTORY_K_v4):
                src = i - (HISTORY_K_v4 - 1 - k)
                if src < 0 or ep_idx[src] != cur_ep:
                    src = i
                    while src - 1 >= 0 and ep_idx[src - 1] == cur_ep:
                        src -= 1
                X[i, k*per_frame_dim:(k+1)*per_frame_dim] = per_frame[src]

        # Targets: goal-relative actions
        chunks = np.zeros((N, ACTION_CHUNK_v4, 2), dtype=np.float32)
        for i in range(N):
            ep_end = i
            while ep_end + 1 < N and ep_idx[ep_end + 1] == ep_idx[i]:
                ep_end += 1
            for k in range(ACTION_CHUNK_v4):
                src = min(i + k, ep_end)
                chunks[i, k] = action[src] - goal_pf_xy[i]
        Y = chunks.reshape(N, -1)

        out = pd.DataFrame({"episode": ep_idx, "frame": frame_idx})
        for i in range(X.shape[1]):
            out[f"x{i}"] = X[:, i]
        for i in range(Y.shape[1]):
            out[f"y{i}"] = Y[:, i]
        return out
if DATA_V4_PATH.exists():
        training_v4 = pd.read_parquet(DATA_V4_PATH)
        print(f"loaded cached v4 ({len(training_v4)} rows)")
    else:
        print("building v4 (keypoint features)...")
        training_v4 = build_v4()
        training_v4.to_parquet(DATA_V4_PATH, index=False)
        print(f"saved {len(training_v4)} rows")
X4_COLS = [c for c in training_v4.columns if c.startswith("x")]
Y4_COLS = [c for c in training_v4.columns if c.startswith("y")]
print(f"X={len(X4_COLS)}D ({HISTORY_K_v4} frames x 34 features), Y={len(Y4_COLS)}D")
training_v4.head()

In [ ]:
_r4 = np.random.RandomState(42)
_eps4 = sorted(training_v4.episode.unique())
_r4.shuffle(_eps4)
_split4 = int(0.8 * len(_eps4))
train_eps_v4 = set(_eps4[:_split4])
val_eps_v4 = set(_eps4[_split4:])
_tm4 = training_v4.episode.isin(train_eps_v4)
_vm4 = training_v4.episode.isin(val_eps_v4)
X_train_v4 = training_v4.loc[_tm4, X4_COLS].to_numpy().astype(np.float32)
Y_train_v4 = training_v4.loc[_tm4, Y4_COLS].to_numpy().astype(np.float32)
X_val_v4 = training_v4.loc[_vm4, X4_COLS].to_numpy().astype(np.float32)
Y_val_v4 = training_v4.loc[_vm4, Y4_COLS].to_numpy().astype(np.float32)
X_mean_v4 = X_train_v4.mean(0)
X_std_v4 = X_train_v4.std(0) + 1e-6
Y_mean_v4 = Y_train_v4.mean(0)
Y_std_v4 = Y_train_v4.std(0) + 1e-6
X_train_v4_n = (X_train_v4 - X_mean_v4) / X_std_v4
Y_train_v4_n = (Y_train_v4 - Y_mean_v4) / Y_std_v4
X_val_v4_n = (X_val_v4 - X_mean_v4) / X_std_v4
Y_val_v4_n = (Y_val_v4 - Y_mean_v4) / Y_std_v4
print(f"train: {len(X_train_v4)} frames ({len(train_eps_v4)} eps)")
print(f"val:   {len(X_val_v4)} frames ({len(val_eps_v4)} eps)")
print(f"X={X_train_v4.shape[1]}D, Y={Y_train_v4.shape[1]}D")

In [ ]:
import torch as _torch_v4
from torch import nn as _nn_v4
class PolicyV4(_nn_v4.Module):
        def __init__(self, in_dim, out_dim, hidden=768, depth=6, dropout=0.15):
            super().__init__()
            layers = [_nn_v4.Linear(in_dim, hidden), _nn_v4.GELU(), _nn_v4.Dropout(dropout)]
            for _ in range(depth - 1):
                layers += [_nn_v4.Linear(hidden, hidden), _nn_v4.GELU(), _nn_v4.Dropout(dropout)]
            layers += [_nn_v4.Linear(hidden, out_dim)]
            self.net = _nn_v4.Sequential(*layers)
        def forward(self, x):
            return self.net(x)
def _train_v4():
        import time as _tt
        import copy as _cp
        _torch_v4.manual_seed(0)
        dev = "mps" if _torch_v4.backends.mps.is_available() else "cpu"
        print(f"device: {dev}")
        m = PolicyV4(X_train_v4.shape[1], Y_train_v4.shape[1]).to(dev)
        nparams = sum(p.numel() for p in m.parameters())
        print(f"params: {nparams:,}")
        EP = 100
        BS = 1024
        opt = _torch_v4.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
        sch = _torch_v4.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
        Xtr = _torch_v4.tensor(X_train_v4_n).to(dev); Ytr = _torch_v4.tensor(Y_train_v4_n).to(dev)
        Xv = _torch_v4.tensor(X_val_v4_n).to(dev); Yv = _torch_v4.tensor(Y_val_v4_n).to(dev)
        hist = []; best_val = float("inf"); best_state = None
        patience = 20; since_best = 0
        t0 = _tt.time()
        for ep in range(EP):
            m.train()
            perm = _torch_v4.randperm(len(Xtr), device=dev)
            ls = []
            for i in range(0, len(perm), BS):
                idx = perm[i:i+BS]
                p = m(Xtr[idx])
                L = ((p - Ytr[idx]) ** 2).mean()
                opt.zero_grad(); L.backward(); opt.step()
                ls.append(L.item())
            sch.step()
            m.eval()
            with _torch_v4.no_grad():
                vp = m(Xv)
                vmse = ((vp - Yv) ** 2).mean().item()
                vp_dn = vp.cpu().numpy() * Y_std_v4 + Y_mean_v4
                vrmse_env = float(np.sqrt(((vp_dn - Y_val_v4) ** 2).mean()))
                vrmse_px = vrmse_env * 96.0 / 512.0
            tr = float(np.mean(ls))
            hist.append({"epoch": ep+1, "train_mse": tr, "val_mse": vmse, "val_rmse_env": vrmse_env, "val_rmse_px": vrmse_px})
            marker = ""
            if vmse < best_val:
                best_val = vmse; best_state = _cp.deepcopy(m.state_dict()); since_best = 0; marker = "  ★"
            else:
                since_best += 1
            if ep < 5 or ep % 2 == 0 or marker:
                print(f"ep {ep+1:3d}: tr={tr:.4f}  v={vmse:.4f}  RMSE={vrmse_env:.1f}env ({vrmse_px:.2f}px){marker}")
            if since_best >= patience:
                print(f"early stop at {ep+1}")
                break
        m.load_state_dict(best_state)
        print(f"\ntrain time: {_tt.time()-t0:.1f}s  best val RMSE: {min(h['val_rmse_env'] for h in hist):.1f}env / {min(h['val_rmse_px'] for h in hist):.2f}px")
        import pickle as _pk
        _torch_v4.save({"state_dict": m.state_dict(), "in_dim": X_train_v4.shape[1], "out_dim": Y_train_v4.shape[1]},
                       "/Users/dimavremenko/Desktop/hackathon-pydata/model_v4.pt")
        with open("/Users/dimavremenko/Desktop/hackathon-pydata/norm_v4.pkl", "wb") as _f:
            _pk.dump({"X_mean": X_mean_v4, "X_std": X_std_v4, "Y_mean": Y_mean_v4, "Y_std": Y_std_v4,
                      "history_k": 4, "action_chunk": 8, "pixel_to_env": 512.0/96.0}, _f)
        return m, hist, dev
model_v4, _history_v4, nn_device_v4 = _train_v4()
history_v4_df = pd.DataFrame(_history_v4)
history_v4_df.tail(8)

In [ ]:
import sys as _sysv4
for _m in [_m for _m in list(_sysv4.modules) if _m.startswith(("pymunk", "gym_pusht"))]:
        del _sysv4.modules[_m]
import gymnasium as _gymv4
import gym_pusht as _gpv4
import collections as _colv4
import time as _timev4
import torch as _torch_v4_eval
import pandas as _pdv4
_goal_df_v4 = _pdv4.read_parquet("/Users/dimavremenko/Desktop/hackathon-pydata/goal_poses.parquet")
ROLLOUT_GOAL_ENV_v4 = np.array([
        _goal_df_v4.x.median() * 512.0/96.0,
        _goal_df_v4.y.median() * 512.0/96.0
    ])
ROLLOUT_GOAL_THETA_v4 = np.deg2rad(_goal_df_v4.theta_deg.median())
T_KP_ENV = np.array([
        [0.0, 0.0],
        [-11.34, -5.77],
        [12.66, -5.77],
        [0.66, 14.23],
    ], dtype=np.float32) * (512.0 / 96.0)
def _kps_at(cx, cy, theta):
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        R = np.array([[cos_t, -sin_t], [sin_t, cos_t]], dtype=np.float32)
        out = T_KP_ENV @ R.T
        out[:, 0] += cx; out[:, 1] += cy
        return out
def _build_feats_v4(history, goal_xy, goal_theta):
        """history: list of (pusher_xy, block_xy, block_theta_rad)"""
        goal_kps = _kps_at(goal_xy[0], goal_xy[1], goal_theta)
        feats = []
        for pusher, block, btheta in history:
            block_kps = _kps_at(block[0], block[1], btheta)
            pusher_rel = pusher - goal_xy
            block_kps_rel = block_kps - goal_xy
            goal_kps_rel = goal_kps - goal_xy
            pusher_to_block = block_kps - pusher
            block_to_goal = goal_kps - block_kps
            feats.extend(pusher_rel.flatten())
            feats.extend(block_kps_rel.flatten())
            feats.extend(goal_kps_rel.flatten())
            feats.extend(pusher_to_block.flatten())
            feats.extend(block_to_goal.flatten())
        return np.array(feats, dtype=np.float32)
def _predict_v4(model, history, goal_xy, goal_theta):
        feats = _build_feats_v4(history, goal_xy, goal_theta)
        feats_n = (feats - X_mean_v4) / X_std_v4
        with _torch_v4_eval.no_grad():
            pred = model(_torch_v4_eval.tensor(feats_n).unsqueeze(0).to(nn_device_v4))
        pred_dn = pred.cpu().numpy().reshape(-1) * Y_std_v4 + Y_mean_v4
        return pred_dn.reshape(8, 2) + goal_xy
def run_episode_v4(model, seed, max_steps=300, execute_per_chunk=4, use_env_goal=False):
        env = _gymv4.make("gym_pusht/PushT-v0", obs_type="state")
        obs, info = env.reset(seed=seed)
        if use_env_goal:
            goal_xy = info["goal_pose"][:2]; goal_theta = info["goal_pose"][2]
        else:
            goal_xy = ROLLOUT_GOAL_ENV_v4; goal_theta = ROLLOUT_GOAL_THETA_v4
        pusher = obs[:2].copy(); block_xy = obs[2:4].copy(); block_theta = obs[4]
        hist = _colv4.deque(maxlen=4)
        for _ in range(4):
            hist.append((pusher.copy(), block_xy.copy(), float(block_theta)))
        max_reward = 0.0; max_coverage = 0.0; success = False; steps = 0
        while steps < max_steps:
            actions = _predict_v4(model, list(hist), goal_xy, goal_theta)
            for k in range(execute_per_chunk):
                if steps >= max_steps: break
                act = np.clip(actions[k], 0.0, 512.0).astype(np.float32)
                obs, reward, term, trunc, info = env.step(act)
                steps += 1
                max_reward = max(max_reward, float(reward))
                max_coverage = max(max_coverage, float(info.get("coverage", reward)))
                if info.get("is_success", False): success = True
                pusher = obs[:2]; block_xy = obs[2:4]; block_theta = obs[4]
                hist.append((pusher.copy(), block_xy.copy(), float(block_theta)))
                if term or trunc or success: break
            if term or trunc or success: break
        env.close()
        return success, max_reward, max_coverage, steps
for _seed in [0, 1, 5, 7, 42]:
        _s, _r, _c, _n = run_episode_v4(model_v4, seed=_seed, execute_per_chunk=4, use_env_goal=False)
        print(f"smoke seed={_seed}: success={_s}, max_r={_r:.3f}, max_cov={_c:.3f}, steps={_n}")
print("\nrunning 50 rollouts (v4, trained goal)...")
_t0 = _timev4.time()
_results_v4 = []
for _seed in range(50):
        _s, _r, _c, _n = run_episode_v4(model_v4, seed=_seed, max_steps=300, execute_per_chunk=4, use_env_goal=False)
        _results_v4.append({"seed": _seed, "success": _s, "max_reward": _r, "max_coverage": _c, "steps": _n})
        if (_seed + 1) % 10 == 0:
            _df = pd.DataFrame(_results_v4)
            print(f"  {_seed+1}/50: success={_df.success.mean()*100:.0f}%, mean cov={_df.max_coverage.mean():.2f}")
rollout_v4_df = pd.DataFrame(_results_v4)
print(f"\nFINAL v4: success rate = {rollout_v4_df.success.mean()*100:.0f}% ({rollout_v4_df.success.sum()}/50)")
print(f"          mean max_cov = {rollout_v4_df.max_coverage.mean():.2f}")
print(f"          cov > 0.95: {(rollout_v4_df.max_coverage > 0.95).sum()}/50")
print(f"          cov > 0.70: {(rollout_v4_df.max_coverage > 0.70).sum()}/50")
print(f"          cov > 0.50: {(rollout_v4_df.max_coverage > 0.50).sum()}/50")
print(f"          time: {_timev4.time() - _t0:.1f}s")
rollout_v4_df.head()

## Takeaway

The Powell fit recovers the T's pose at ~IoU 0.9+ on the sample frames above, with no learned model and no GPU. The 'cheat' is recognizing that the perception problem in Push-T is a 4-parameter geometric fit dressed up as an image problem — and that diffusion policies are doing far more work than the task actually requires.
